# Complete LSTM Sentiment Analysis - Colab GPU Workflow

Run all requests a NEW training run after every gate succeeds. A completed owner-run experiment has been audited: Accuracy 86.09%, Macro-F1 0.8607. See [README](../README.md) for the saved results and limitations. The original 86.29% is historical only.

Use a fresh free GPU runtime. Do not purchase credits or storage. Upload the project ZIP and separate CSV when prompted. No GitHub access or API keys are required. No new training was executed during the local results audit.


## 1. Upload the project ZIP

Select `lstm_sentiment_colab.zip`. The archive excludes the CSV, weights and original notebook. Bootstrap validates paths and file hashes, then extracts into a fresh `/content/lstm_colab_.../lstm_sentiment_analysis` folder. The manifest verifies integrity, not authorship.


In [ ]:
import os
import sys
import io
import json
import hashlib
import tempfile
import zipfile
from pathlib import Path, PurePosixPath
from google.colab import files

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one project ZIP')
archive_name, archive_bytes = next(iter(uploaded.items()))
if not archive_name.lower().endswith('.zip'):
    raise ValueError('Expected the project ZIP; upload the CSV in the later dataset cell')
destination = Path(tempfile.mkdtemp(prefix='lstm_colab_', dir='/content'))
prefix = 'lstm_sentiment_analysis'
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise ValueError('Duplicate archive paths')
    for info in archive.infolist():
        path = PurePosixPath(info.filename)
        if path.is_absolute() or '..' in path.parts or not path.parts or path.parts[0] != prefix:
            raise ValueError('Unsafe archive path')
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Archive symlinks are not allowed')
    manifest = json.loads(archive.read(prefix + '/MANIFEST.json'))
    if manifest.get('format_version') != 1:
        raise ValueError('Unsupported manifest version')
    expected_names = {prefix + '/' + name for name in manifest['files']} | {prefix + '/MANIFEST.json'}
    if set(names) != expected_names:
        raise ValueError('Archive contents do not match manifest')
    archive.extractall(destination)
ROOT = destination / prefix
for name, expected in manifest['files'].items():
    content = (ROOT / name).read_bytes()
    if len(content) != expected['bytes'] or hashlib.sha256(content).hexdigest() != expected['sha256']:
        raise ValueError('File checksum mismatch: ' + name)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('Verified project:', ROOT)
print('ZIP SHA-256:', hashlib.sha256(archive_bytes).hexdigest())
del uploaded, archive_bytes

SOURCE_MANIFEST = manifest


## 2. Settings

Default `RUN_FULL_TRAINING=True` means Run all continues through full training after checks. Set False for verification only. Keep seed/config unchanged for the default experiment.

Default outputs use temporary Colab storage. Optional `PERSIST_TO_DRIVE=True` mounts your own Drive interactively and preserves logs/checkpoints; the assistant never logs in. No paid storage is required or purchased. A deleted runtime cannot run cleanup or initiate a download. Drive persistence protects already-written files but does **not** implement optimizer/RNG resume. Start a new run after a full reset.


In [ ]:
RUN_FULL_TRAINING = True
PERSIST_TO_DRIVE = False
EXISTING_DATA_PATH = None  # Optional verified CSV already present in this Colab runtime.
TRAIN_CONFIG = {
    'seed': 42, 'max_length': 200, 'max_vocab': 20_000, 'min_frequency': 2,
    'embedding_dim': 100, 'hidden_dim': 128, 'dropout': 0.5,
    'batch_size': 64, 'learning_rate': 0.001, 'epochs': 10, 'patience': 3,
}


## 3. Execute the complete workflow

The controller performs these stages sequentially:

1. Install Colab-only requirements and verify Python/NumPy/matplotlib and existing CUDA-enabled PyTorch. No CPU fallback.
2. Run **all unit tests** in a fresh subprocess, requiring zero failures/errors/skips and all six original model tests.
3. Run one synthetic GPU optimization step, checking forward, padding and portable save/restore. This is not IMDb training.
4. Prompt for `IMDB Dataset.csv`, verify schema, missing values, labels, 50,000 records and the inspected SHA-256. Existing files are read, not overwritten.
5. Recheck source and dataset hashes; train the improved model on training only. Select and early-stop using validation loss only.
6. Restore the selected checkpoint, then evaluate test once with fixed threshold 0.5.
7. Save model/vocabulary/config, metrics, history, PNG curves/matrix, CSV and Markdown report.
8. Build and download the results ZIP. A caught error/interrupt produces failed/interrupted diagnostics when possible.

Training persists a diagnostic best checkpoint and history after each epoch. There is no exact resume. A live browser disconnect may leave the subprocess running; reconnect and inspect logs before launching another run. If the runtime was killed, partial outputs are never labeled a completed evaluation. On GPU OOM, retain diagnostics and start a new experiment with smaller batch size; never choose settings based on test scores.


In [ ]:
from datetime import datetime, timezone
from sentiment.colab_workflow import run_colab_workflow
from sentiment.workflow import export_results

if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    runs_root = Path('/content/drive/MyDrive/LSTM_Sentiment_Analysis/runs')
else:
    runs_root = Path('/content/lstm_runs')
runs_root.mkdir(parents=True, exist_ok=True)
RUN_DIR = runs_root / datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
print('Planned run:', RUN_DIR)
RESULT_ZIP = run_colab_workflow(ROOT, SOURCE_MANIFEST, RUN_DIR, TRAIN_CONFIG,
                                train=RUN_FULL_TRAINING, existing_data_path=EXISTING_DATA_PATH)


## 4. Output paths and recovery

The console prints exact paths. Default layout:

```text
/content/lstm_runs/run_<UTC>/
  status.json, config.json, source_manifest.json
  environment.json, tests.json, tests.log, smoke.json, smoke.log
  dataset.json, setup.log, training.log, workflow_error.log (on error)
  model/model.pt, metadata.json, metrics.json, history.json, status.json
  model/partial/best_weights.pt, metadata.json, history.json
  figures/loss.png, accuracy.png, confusion_matrix.png
  reports/results.md, metrics.csv, provenance.json
/content/lstm_runs/run_<UTC>_results.zip
```

With optional Drive, use `MyDrive/LSTM_Sentiment_Analysis/runs` instead. The results ZIP includes output files and a status/hash manifest, final model and any diagnostic partial checkpoint, but never the input CSV or credentials. The portable vocabulary and preprocessing/configuration are in `model/metadata.json`.

`complete` workflow status plus `trained` model status denotes completed evaluation. `running`, `failed`, `interrupted` and `verified_not_trained` do not. A lost temporary runtime cannot be recovered. If Drive is enabled, restart all checks and begin a new run; do not label an old partial checkpoint as a completed run.


## 5. Optional recovery download — no training

Use only if the automatic browser download failed and files still exist. For a persisted earlier run, assign its explicit path to RUN_DIR first. This cell only rebuilds/downloads output artifacts.


In [ ]:
# Recovery only: RUN_DIR = Path('/content/drive/MyDrive/LSTM_Sentiment_Analysis/runs/run_<UTC>')
if 'RUN_DIR' not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError('Run files unavailable; a deleted temporary runtime cannot be recovered.')
RESULT_ZIP = Path(RUN_DIR).parent / (Path(RUN_DIR).name + '_results.zip')
export_results(RUN_DIR, RESULT_ZIP)
files.download(str(RESULT_ZIP))


## Interpretation and limitations

Reports use only the newly completed improved run. Original 86.29% is not attributed to it and is not a directly comparable benchmark. Limitations: English regex tokens, truncation, one seed, custom split, no near-duplicate/movie-group control, no calibration or out-of-domain validation. Review measured metrics, split fingerprints and environment before making claims. No full training or GPU success was established during local preparation.
